In [1]:
import pandas as pd 
import numpy as np 

In [2]:
movies = pd.read_csv(r'../datasets/movies.csv')
ratings = pd.read_csv(r'../datasets/ratings.csv')
tags = pd.read_csv(r'../datasets/tags.csv')

## Movies

In [142]:
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [143]:
movies['year'] = movies['title'].str.extract(r'\((\d{4})\)').copy()

In [145]:
movies

,movieId,title,genres,year
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1995
1,2,Jumanji (1995),Adventure|Children|Fantasy,1995
2,3,Grumpier Old Men (1995),Comedy|Romance,1995
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,1995
4,5,Father of the Bride Part II (1995),Comedy,1995
...,...,...,...,...
9737,193581,Black Butler: Book of the Atlantic (2017),Action|Animation|Comedy|Fantasy,2017
9738,193583,No Game No Life: Zero (2017),Animation|Comedy|Fantasy,2017
9739,193585,Flint (2017),Drama,2017
9740,193587,Bungo Stray Dogs: Dead Apple (2018),Action|Animation,2018


In [146]:
movies.info()

<class 'pandas.DataFrame'>
RangeIndex: 9742 entries, 0 to 9741
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   movieId  9742 non-null   int64
 1   title    9742 non-null   str  
 2   genres   9742 non-null   str  
 3   year     9729 non-null   str  
dtypes: int64(1), str(3)
memory usage: 304.6 KB


In [147]:
# Saving the data 
movies.to_csv(r'..\datasets\new_data_sets\movies.csv',index=False)

## Ratings

In [45]:
ratings

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931
...,...,...,...,...
100831,610,166534,4.0,1493848402
100832,610,168248,5.0,1493850091
100833,610,168250,5.0,1494273047
100834,610,168252,5.0,1493846352


In [46]:
ratings.info()

<class 'pandas.DataFrame'>
RangeIndex: 100836 entries, 0 to 100835
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     100836 non-null  int64  
 1   movieId    100836 non-null  int64  
 2   rating     100836 non-null  float64
 3   timestamp  100836 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 3.1 MB


In [47]:
# first we will Convert Unix timestamp to datetime
ratings['timestamp_dt'] = pd.to_datetime(ratings['timestamp'], unit='s')
ratings['rating_year'] = ratings['timestamp_dt'].dt.year
ratings

,userId,movieId,rating,timestamp,timestamp_dt,rating_year
0,1,1,4.0,964982703,2000-07-30 18:45:03,2000
1,1,3,4.0,964981247,2000-07-30 18:20:47,2000
2,1,6,4.0,964982224,2000-07-30 18:37:04,2000
3,1,47,5.0,964983815,2000-07-30 19:03:35,2000
4,1,50,5.0,964982931,2000-07-30 18:48:51,2000
...,...,...,...,...,...,...
100831,610,166534,4.0,1493848402,2017-05-03 21:53:22,2017
100832,610,168248,5.0,1493850091,2017-05-03 22:21:31,2017
100833,610,168250,5.0,1494273047,2017-05-08 19:50:47,2017
100834,610,168252,5.0,1493846352,2017-05-03 21:19:12,2017


# Popular Movies Recommender data set

In [148]:
# Getting number of ratings applied to each movie and avg_rating applied to each movie by users
popular_movies = ratings.groupby('movieId').agg(
    num_ratings = ('rating','count'),
    avg_rating = ('rating','mean')
).reset_index().copy()
    

In [149]:
# keeping only those movieId's where num_ratings is greater than 200 and avg rating is greater than 3
popular_movies = popular_movies.query('num_ratings >= 150 and avg_rating >= 2.8').copy()

In [150]:
popular_movies = popular_movies.sort_values('avg_rating',ascending=False).copy()

In [54]:
popular_movies_final = pd.merge(popular_movies,movies,on='movieId')

In [56]:
popular_movies_final

,movieId,num_ratings,avg_rating,title,genres,year
0,318,317,4.429022,"Shawshank Redemption, The (1994)",Crime|Drama,1994
1,858,192,4.289062,"Godfather, The (1972)",Crime|Drama,1972
2,2959,218,4.272936,Fight Club (1999),Action|Crime|Drama|Thriller,1999
3,50,204,4.237745,"Usual Suspects, The (1995)",Crime|Mystery|Thriller,1995
4,260,251,4.231076,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Sci-Fi,1977
5,527,220,4.225000,Schindler's List (1993),Drama|War,1993
6,1196,211,4.215640,Star Wars: Episode V - The Empire Strikes Back...,Action|Adventure|Sci-Fi,1980
7,1198,200,4.207500,Raiders of the Lost Ark (Indiana Jones and the...,Action|Adventure,1981
8,296,307,4.197068,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller,1994
9,2571,278,4.192446,"Matrix, The (1999)",Action|Sci-Fi|Thriller,1999


In [64]:
# Saving the data 
popular_movies_final.to_csv(r'..\datasets\new_data_sets\popular_movies.csv',index=False)

# Collaborative Filtering Recommender 

In [3]:
colab_filt_df = ratings.copy()

In [4]:
colab_filt_df['userId'].nunique()

610

In [5]:
colab_filt_df

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931
...,...,...,...,...
100831,610,166534,4.0,1493848402
100832,610,168248,5.0,1493850091
100833,610,168250,5.0,1494273047
100834,610,168252,5.0,1493846352


In [6]:
# Getting only those users who have rated at least 30 movies 
x = colab_filt_df.groupby('userId')['rating'].count() > 30
# choosing users that specify the above condition and getting there id 
filtered_users = x[x].index

In [7]:
# now consider only filtered_users in ratings frame
filtered_ratings = ratings[ratings['userId'].isin(filtered_users)]

In [8]:
ratings

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931
...,...,...,...,...
100831,610,166534,4.0,1493848402
100832,610,168248,5.0,1493850091
100833,610,168250,5.0,1494273047
100834,610,168252,5.0,1493846352


In [9]:
print(filtered_ratings['userId'].nunique())
print(filtered_ratings['movieId'].nunique())

498
9689


In [10]:
# so we left with 498 users who rated over 9689
# Now we create our final collaborative filtering data frame

"""
Below function work like it place the movieId.unique() in first column and for every userId.unque() it create a column and the values of that columns are the rating the user provided to that movie

"""
final_colab_df = filtered_ratings.pivot_table(index = 'movieId',columns='userId',values='rating')
final_colab_df

userId,1,3,4,5,6,7,8,9,10,11,...,601,602,603,604,605,606,607,608,609,610
movieId,,,,,,,,,,,,,,,,,,,,,
1,4.0,NaN,NaN,4.0,NaN,4.5,NaN,NaN,NaN,NaN,...,4.0,NaN,4.0,3.0,4.0,2.5,4.0,2.5,3.0,5.0
2,NaN,NaN,NaN,NaN,4.0,NaN,4.0,NaN,NaN,NaN,...,NaN,4.0,NaN,5.0,3.5,NaN,NaN,2.0,NaN,NaN
3,4.0,NaN,NaN,NaN,5.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,NaN,NaN
4,NaN,NaN,NaN,NaN,3.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,5.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,3.0,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
193581,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
193583,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
193585,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
# replacing missing value with 0
final_colab_df = final_colab_df.fillna(0,inplace=True)
final_colab_df

userId,1,3,4,5,6,7,8,9,10,11,...,601,602,603,604,605,606,607,608,609,610
movieId,,,,,,,,,,,,,,,,,,,,,
1,4.0,0.0,0.0,4.0,0.0,4.5,0.0,0.0,0.0,0.0,...,4.0,0.0,4.0,3.0,4.0,2.5,4.0,2.5,3.0,5.0
2,0.0,0.0,0.0,0.0,4.0,0.0,4.0,0.0,0.0,0.0,...,0.0,4.0,0.0,5.0,3.5,0.0,0.0,2.0,0.0,0.0
3,4.0,0.0,0.0,0.0,5.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0
4,0.0,0.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,0.0,0.0,5.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
193581,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
193583,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
193585,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [3]:
from surprise import SVD
from surprise import Dataset,Reader,accuracy
from surprise.model_selection import GridSearchCV
from surprise.model_selection import train_test_split
from surprise import dump
from surprise import SVD

In [4]:
import pandas as pd
import time

# 1. Define a unique User ID
MY_USER_ID = 99999

# 2. Define your ratings (movieId, rating)
my_custom_ratings = [
    # Shawshank Redemption (5.0)
    {"userId": MY_USER_ID, "movieId": 318, "rating": 5.0},
    
    # Harry Potter Series (5.0)
    {"userId": MY_USER_ID, "movieId": 4896, "rating": 5.0},
    {"userId": MY_USER_ID, "movieId": 5816, "rating": 5.0},
    {"userId": MY_USER_ID, "movieId": 8368, "rating": 5.0},
    {"userId": MY_USER_ID, "movieId": 40815, "rating": 5.0},
    {"userId": MY_USER_ID, "movieId": 54001, "rating": 5.0},
    {"userId": MY_USER_ID, "movieId": 69844, "rating": 5.0},
    {"userId": MY_USER_ID, "movieId": 81834, "rating": 5.0},
    {"userId": MY_USER_ID, "movieId": 88125, "rating": 5.0},
    
    # Pirates of the Caribbean Series (4.0)
    {"userId": MY_USER_ID, "movieId": 6539, "rating": 4.0},
    {"userId": MY_USER_ID, "movieId": 45722, "rating": 4.0},
    {"userId": MY_USER_ID, "movieId": 53125, "rating": 4.0},
    {"userId": MY_USER_ID, "movieId": 87232, "rating": 4.0},
    
    # Toy Story Series (3.0)
    {"userId": MY_USER_ID, "movieId": 1, "rating": 3.0},
    {"userId": MY_USER_ID, "movieId": 3114, "rating": 3.0},
    {"userId": MY_USER_ID, "movieId": 78499, "rating": 3.0},
]
# 3. Create DataFrame
my_ratings_df = pd.DataFrame(my_custom_ratings)
my_ratings_df['timestamp'] = int(time.time())


In [5]:
new_ratings = pd.concat([ratings,my_ratings_df],ignore_index=True)

In [6]:
new_ratings.tail()

,userId,movieId,rating,timestamp
100847,99999,53125,4.0,1791378692
100848,99999,87232,4.0,1791378692
100849,99999,1,3.0,1791378692
100850,99999,3114,3.0,1791378692
100851,99999,78499,3.0,1791378692


In [7]:
reader = Reader(
    rating_scale=(
        ratings['rating'].min(),
        ratings['rating'].max()
    )
)

In [8]:
data_set = Dataset.load_from_df(ratings[['userId','movieId','rating']],reader)

In [9]:
train_set,test_set = train_test_split(data_set,test_size=0.2,random_state=42)

In [25]:
parameter_grid = {
    'n_factors':[20,30,40],
    'n_epochs':[100,200,50],
    'lr_all':[0.1,0.1,0.01,0.5,0.05],
    'reg_all':[0.1,0.01,0.5,0.05],
}

In [28]:
gs = GridSearchCV(
    SVD,
    param_grid=parameter_grid,
    measures=['rmse','mae'],
    n_jobs = -1,
    cv=3, # it divides the data into 3 parts 2 for training and 
    joblib_verbose=1
)

In [30]:
gs.fit(data_set)

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=-1)]: Done  42 tasks      | elapsed:  1.5min
[Parallel(n_jobs=-1)]: Done 192 tasks      | elapsed:  7.3min
[Parallel(n_jobs=-1)]: Done 442 tasks      | elapsed: 23.6min
[Parallel(n_jobs=-1)]: Done 540 out of 540 | elapsed: 27.0min finished


In [80]:
for param,value in gs.best_params['rmse'].items():
    print(f'{param}:{value}')

n_factors:40
n_epochs:50
lr_all:0.01
reg_all:0.1


In [10]:
svd_model = SVD(
    n_factors=60,
    lr_all=0.01,
    reg_all=0.1,
    random_state=42,
    n_epochs=50
)
svd_model.fit(train_set)

In [11]:
predictions = svd_model.test(test_set)
RMSE = accuracy.rmse(predictions)

RMSE: 0.8581


In [12]:
uid = 99999
mid = 72998


In [13]:
user_rat = svd_model.predict(uid,mid)

In [14]:
print(f'{user_rat.est:0.3}')

3.56


In [134]:
dump.dump('../models/colab_filt_svd',algo=svd_model)

In [15]:
test_model_path = '../models/colab_filt_svd'
ratings_path = '../datasets/ratings.csv'
movies_path = '../datasets/movies.csv'

In [16]:
import sys
from pathlib import Path

# Add project root to sys.path
PROJECT_ROOT = Path(r"C:\My storage\My_Ml_projects\Movie_recommender-Hybrid-")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

# Import directly using the exact filename 'svd_pipe_line.py'
from src.pipelines.svd_pipe_line import SVD_pipeline

print("Successfully imported SVD_pipeline!")

Successfully imported SVD_pipeline!


In [17]:
my_pipeline = SVD_pipeline(test_model_path,ratings_path,movies_path)

In [20]:
movies = my_pipeline.get_n_recommendations(uid=99999,n=100)

In [21]:
movies

,movieId,title,genres,predicted_rating,year
0,1178,Paths of Glory (1957),Drama|War,4.59,1957.0
1,7121,Adam's Rib (1949),Comedy|Romance,4.54,1949.0
2,177593,"Three Billboards Outside Ebbing, Missouri (2017)",Crime|Drama,4.51,2017.0
3,6460,"Trial, The (Procès, Le) (1962)",Drama,4.48,1962.0
4,2202,Lifeboat (1944),Drama|War,4.43,1944.0
...,...,...,...,...,...
95,8405,Hour of the Wolf (Vargtimmen) (1968),Drama|Horror,4.20,1968.0
96,3379,On the Beach (1959),Drama,4.20,1959.0
97,4956,"Stunt Man, The (1980)",Action|Adventure|Comedy|Drama|Romance|Thriller,4.20,1980.0
98,27397,Joint Security Area (Gongdong gyeongbi guyeok ...,Crime|Drama|Mystery|Thriller|War,4.20,2000.0
